# Unit 5 — 03: Alerting and Incident Management for ML Models

**What you will do:** Build an `AlertManager` class that checks four conditions, generate synthetic logs with injected faults, run the manager, and walk through an incident response runbook.

**Why it matters:** Without alerts, you discover outages when users complain — hours after the fact. Automated alerting is the difference between a 5-minute response and a 5-hour outage.

**How to run:** Python 3.10+. Run cells in order.

---

## Section 1 — What Triggers an Alert

Four standard alert conditions for production ML services:

| Condition | Threshold | Implication |
|---|---|---|
| **Error rate** | > 1% of requests | Model or input validation is broken |
| **p99 latency** | > 500 ms | Model is too slow for SLO |
| **Prediction confidence** | < 0.6 average | Model is uncertain — likely distribution shift |
| **Dead service** | Zero predictions in last 5 min | Service has crashed or is not receiving traffic |

These thresholds are starting points. Real systems tune them based on observed baseline behavior.

## Section 2 — AlertManager Class

In [1]:
# Build the AlertManager: four independent checks (error rate, p99 latency,
# mean confidence, service liveness), each returning an alert string when its
# threshold is breached, or None when the metric is healthy.
import numpy as np
from typing import List, Dict, Optional
from datetime import datetime, timedelta, timezone


class AlertManager:
    """Checks production logs against alert thresholds."""

    def check_error_rate(
        self, logs: List[Dict], threshold: float = 0.01
    ) -> Optional[str]:
        """Alert if error rate exceeds threshold (default 1%)."""
        if not logs:
            return None
        n_errors = sum(1 for r in logs if r.get("error", False))
        rate = n_errors / len(logs)
        if rate > threshold:
            return (
                f"ALERT [error_rate]: {rate:.2%} > threshold {threshold:.2%} "
                f"({n_errors}/{len(logs)} errors)"
            )
        return None

    def check_latency(
        self, logs: List[Dict], p99_threshold_ms: float = 500.0
    ) -> Optional[str]:
        """Alert if p99 latency exceeds threshold."""
        latencies = [r["latency_ms"] for r in logs if not r.get("error", False)]
        if not latencies:
            return None
        p99 = float(np.percentile(latencies, 99))
        if p99 > p99_threshold_ms:
            return (
                f"ALERT [latency_p99]: {p99:.1f}ms > threshold {p99_threshold_ms:.1f}ms"
            )
        return None

    def check_confidence(
        self, logs: List[Dict], threshold: float = 0.6
    ) -> Optional[str]:
        """Alert if mean prediction confidence drops below threshold."""
        confidences = [
            r["confidence"] for r in logs
            if not r.get("error", False) and "confidence" in r
        ]
        if not confidences:
            return None
        mean_conf = float(np.mean(confidences))
        if mean_conf < threshold:
            return (
                f"ALERT [confidence]: mean={mean_conf:.3f} < threshold {threshold:.3f}"
            )
        return None

    def check_service_alive(
        self, logs: List[Dict], window_minutes: float = 5.0,
        now: Optional[datetime] = None,
    ) -> Optional[str]:
        """Alert if no successful prediction within window_minutes of `now`.

        `now` defaults to the newest timestamp in the log slice, so reviewing a
        HISTORICAL batch of logs asks "was the service alive at the end of that
        window?" — old-but-healthy logs must not trip the alarm. For LIVE
        monitoring, pass now=datetime.now(timezone.utc) to compare against the
        wall clock instead (demonstrated in Section 4).
        """
        if not logs:
            return (
                f"ALERT [dead_service]: No successful predictions in last {window_minutes:.0f} minutes."
            )
        if now is None:
            # Anchor the window to the newest log entry being examined
            now = max(datetime.fromisoformat(r["timestamp"]) for r in logs)
        cutoff = now - timedelta(minutes=window_minutes)
        recent = [
            r for r in logs
            if not r.get("error", False)
            and datetime.fromisoformat(r["timestamp"]) > cutoff
        ]
        if not recent:
            return (
                f"ALERT [dead_service]: No successful predictions in last {window_minutes:.0f} minutes."
            )
        return None

    def run_all_checks(
        self, logs: List[Dict], now: Optional[datetime] = None
    ) -> List[str]:
        """Run all checks and return a list of triggered alerts.

        `now` is forwarded to check_service_alive (None = anchor the liveness
        window to the newest log timestamp)."""
        alerts = []
        for check_fn in [
            self.check_error_rate,
            self.check_latency,
            self.check_confidence,
            lambda l: self.check_service_alive(l, now=now),
        ]:
            result = check_fn(logs)
            if result:
                alerts.append(result)
        return alerts


manager = AlertManager()
print("AlertManager ready with 4 checks.")

AlertManager ready with 4 checks.


## Section 3 — Generate Test Logs with Injected Faults

- First 1000 requests: normal (low latency, high confidence, no errors)
- Then inject 50 errors and 30 slow requests

In [2]:
# Generate a synthetic production log: 1000 requests over 30 minutes, then
# inject known faults (errors + slow requests) so we can verify the alerts
# fire for exactly the problems we planted.
rng = np.random.default_rng(99)


def make_log(
    timestamp: datetime,
    latency_ms: float,
    confidence: float,
    error: bool = False,
) -> Dict:
    return {
        "timestamp": timestamp.isoformat(),
        "latency_ms": round(latency_ms, 2),
        "confidence": round(confidence, 4),
        "error": error,
    }


# Timezone-aware 'now': every log timestamp is anchored to this moment
now = datetime.now(timezone.utc)
logs: List[Dict] = []

# 1000 normal requests spread over the last 30 minutes
for i in range(1000):
    ts = now - timedelta(minutes=30) + timedelta(seconds=i * 1.8)
    lat = float(rng.normal(40, 10))
    conf = float(np.clip(rng.normal(0.85, 0.08), 0.5, 1.0))
    logs.append(make_log(ts, max(5.0, lat), conf))

# Inject 50 errors
error_indices = rng.choice(1000, size=50, replace=False)
for idx in error_indices:
    logs[idx]["error"] = True
    logs[idx]["confidence"] = 0.0

# Inject 30 slow requests (latency > 500ms)
slow_indices = rng.choice(1000, size=30, replace=False)
for idx in slow_indices:
    logs[idx]["latency_ms"] = float(rng.uniform(600, 1200))

print(f"Generated {len(logs)} log entries.")
print(f"  Errors injected:     {sum(1 for r in logs if r['error'])}")
print(f"  Slow requests (>500ms): {sum(1 for r in logs if r['latency_ms'] > 500 and not r['error'])}")

# Show a sample record
import json
print("\nSample log record:")
print(json.dumps(logs[0], indent=2))

Generated 1000 log entries.
  Errors injected:     50
  Slow requests (>500ms): 29

Sample log record:
{
  "timestamp": "2026-08-22T19:52:55.736563+00:00",
  "latency_ms": 40.82,
  "confidence": 0.8128,
  "error": false
}


## Section 4 — Run the AlertManager

Three demonstrations:
1. **Full faulty log** — the injected errors and slow requests must fire alerts.
2. **Clean historical slice** — healthy logs must fire *nothing*, even though they are 15-30 minutes old (the liveness window anchors to the newest timestamp in the slice).
3. **Dead service** — the same old slice evaluated against the *live clock* simulates a service that stopped answering 15 minutes ago, firing the `dead_service` alert on purpose.

In [3]:
# Contrast demo: the full faulty log should fire alerts; a clean slice should not.
print("=" * 60)
print("Running AlertManager on full log (1000 requests)...")
print("=" * 60)

alerts = manager.run_all_checks(logs)

if alerts:
    print(f"\n{len(alerts)} alert(s) fired:\n")
    for a in alerts:
        print(f"  {a}")
else:
    print("\nNo alerts. All metrics within thresholds.")

print()

# Build a clean comparison slice: first 500 requests, errors and injected
# slow requests removed — this is what a healthy service's logs look like
clean_logs = [r for r in logs[:500] if not r["error"]]
for r in clean_logs:
    if r["latency_ms"] > 500:
        r["latency_ms"] = float(rng.normal(40, 10))

# The clean slice is 15-30 minutes OLD. run_all_checks anchors the liveness
# window to the newest timestamp IN the slice, so healthy-but-historical logs
# do not trip the dead-service alarm.
print("Running AlertManager on clean first 500 requests...")
clean_alerts = manager.run_all_checks(clean_logs)
if clean_alerts:
    for a in clean_alerts:
        print(f"  {a}")
else:
    print("  No alerts on clean data (as expected).")

print()

# Now fire the dead-service alert ON PURPOSE: evaluate the same old slice
# against the LIVE wall clock. The newest entry in it is ~15 minutes old, so
# from "right now" the service looks like it stopped answering — exactly the
# situation this check exists to catch.
print("Evaluating the same old slice against the live clock (service stopped 15 min ago):")
stalled_alert = manager.check_service_alive(clean_logs, now=datetime.now(timezone.utc))
print(f"  {stalled_alert}")


Running AlertManager on full log (1000 requests)...

2 alert(s) fired:

  ALERT [error_rate]: 5.00% > threshold 1.00% (50/1000 errors)
  ALERT [latency_p99]: 1037.7ms > threshold 500.0ms

Running AlertManager on clean first 500 requests...
  No alerts on clean data (as expected).

Evaluating the same old slice against the live clock (service stopped 15 min ago):
  ALERT [dead_service]: No successful predictions in last 5 minutes.


## Section 5 — Incident Response Runbook

When an alert fires, follow a structured runbook. These are the standard first-response steps:

### High Latency Alert
1. Check model size — did a new, larger model get deployed?
2. Check batch size — is the service receiving more concurrent requests than expected?
3. Check hardware utilization — is the GPU/CPU saturated?
4. Check for data preprocessing bottlenecks before the model call.

### High Error Rate Alert
1. Pull the last 10 error logs — what exception is being raised?
2. Check input validation — are clients sending malformed payloads?
3. If the error started after a deployment: roll back the model artifact.
4. If the error is in a dependency (database, feature store): escalate to infrastructure team.

### Confidence Drop Alert
1. Run drift detection on the last N hours of input features (see notebook 04).
2. If drift is confirmed: trigger retraining pipeline (see notebook 02).
3. If no drift detected: check for data pipeline issues (wrong feature values being sent).

### Dead Service Alert
1. Hit the `/health` endpoint directly — does it respond?
2. In Kubernetes: `kubectl get pods -n ml-serving` — is the pod running?
3. Check pod logs: `kubectl logs <pod-name> -n ml-serving --tail=100`
4. If pod is crash-looping: describe the pod for the OOMKilled / error reason.

## Section 6 — Simulated Webhook Notification

In [4]:
import json
from typing import List


def send_webhook_alert(alerts: List[str], webhook_url: str = "https://hooks.slack.com/...") -> None:
    """
    In production: POST to a Slack webhook or PagerDuty API.
    Here we simulate the payload that would be sent.
    """
    payload = {
        "text": "*ML Model Alert*",
        "attachments": [
            {
                "color": "danger",
                "fields": [
                    {"title": f"Alert {i+1}", "value": alert, "short": False}
                    for i, alert in enumerate(alerts)
                ],
            }
        ],
    }
    # In production: requests.post(webhook_url, json=payload)
    print(f"[SIMULATED] POST to {webhook_url}")
    print(json.dumps(payload, indent=2))


if alerts:
    send_webhook_alert(alerts)
else:
    print("No alerts to send.")

[SIMULATED] POST to https://hooks.slack.com/...
{
  "text": "*ML Model Alert*",
  "attachments": [
    {
      "color": "danger",
      "fields": [
        {
          "title": "Alert 1",
          "value": "ALERT [error_rate]: 5.00% > threshold 1.00% (50/1000 errors)",
          "short": false
        },
        {
          "title": "Alert 2",
          "value": "ALERT [latency_p99]: 1037.7ms > threshold 500.0ms",
          "short": false
        }
      ]
    }
  ]
}


## Summary

The alerting loop for production ML models:

1. **Define thresholds** for error rate, latency, confidence, and liveness
2. **Evaluate continuously** — run checks on a rolling window of recent logs
3. **Fire alerts** via Slack/PagerDuty when thresholds are breached
4. **Follow the runbook** — each alert type has a standard diagnostic sequence
5. **Act** — roll back, retrain, or escalate depending on root cause

## Self-Check (answer before scrolling back up)

1. **What is an SLO and how does it relate to your alert thresholds?**  
   An SLO (Service Level Objective) is a commitment to a performance target, such as "p99 latency < 300ms" or "error rate < 0.5%". Alert thresholds should be set slightly inside the SLO so that you get warned before you actually breach your commitment to users.

2. **What is the first thing you check when error rate spikes?**  
   Read the actual error messages from the last N failed requests. The exception type tells you immediately whether this is a client-side problem (bad input), a model-side problem (inference crash), or an infrastructure problem (database timeout).

3. **Why do you need a 'service alive' check separate from a latency check?**  
   If the service crashes completely, it produces zero requests — so latency metrics simply stop updating. You cannot distinguish "the service is healthy and idle" from "the service is down" using latency alone. A liveness check explicitly detects the absence of traffic.